In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

raw_parquet_path = "/content/drive/MyDrive/CICIDS_raw_merged.parquet"
data = pd.read_parquet(raw_parquet_path)

In [ ]:
data.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max         

In [ ]:
# Renaming the columns by removing leading/trailing whitespace
col_names = {col: col.strip() for col in data.columns}
data.rename(columns = col_names, inplace = True)

In [ ]:
data['Label'].value_counts()

,count
Label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


In [ ]:
dups = data[data.duplicated()]
print(f'Number of duplicates: {len(dups)}')

Number of duplicates: 308381


In [ ]:
data.drop_duplicates(inplace = True)
data.shape


(2522362, 79)

In [ ]:


missing_val = data.isna().sum()
print(missing_val.loc[missing_val > 0])


Flow Bytes/s    353
dtype: int64


In [ ]:
import numpy as np

numeric_cols = data.select_dtypes(include = np.number).columns
inf_count = np.isinf(data[numeric_cols]).sum()
print(inf_count[inf_count > 0])

Flow Bytes/s      1211
Flow Packets/s    1564
dtype: int64


In [ ]:

# Replacing any infinite values (positive or negative) with NaN (not a number)
print(f'Initial missing values: {data.isna().sum().sum()}')

data.replace([np.inf, -np.inf], np.nan, inplace = True)

print(f'Missing values after processing infinite values: {data.isna().sum().sum()}')



Initial missing values: 353
Missing values after processing infinite values: 3128


In [ ]:

missing = data.isna().sum()
print(missing.loc[missing > 0])

Flow Bytes/s      1564
Flow Packets/s    1564
dtype: int64


In [ ]:

med_flow_bytes = data['Flow Bytes/s'].median()
med_flow_packets = data['Flow Packets/s'].median()

print('Median of Flow Bytes/s: ', med_flow_bytes)
print('Median of Flow Packets/s: ', med_flow_packets)


Median of Flow Bytes/s:  3715.0378579999997
Median of Flow Packets/s:  69.742244285


In [ ]:
# Filling missing values with median
data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
data['Flow Packets/s'].fillna(med_flow_packets, inplace = True)


/tmp/ipykernel_408/1961396149.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
/tmp/ipykernel_408/1961396149.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

In [ ]:
print('Number of \'Flow Bytes/s\' missing values:', data['Flow Bytes/s'].isna().sum())
print('Number of \'Flow Packets/s\' missing values:', data['Flow Packets/s'].isna().sum())

Number of 'Flow Bytes/s' missing values: 0
Number of 'Flow Packets/s' missing values: 0


# Mapping Label to attack type

In [ ]:
attack_map = {
    'BENIGN': 'BENIGN',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'PortScan': 'Port Scan',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bot',
    'Web Attack � Brute Force': 'Web Attack',
    'Web Attack � XSS': 'Web Attack',
    'Web Attack � Sql Injection': 'Web Attack',
    'Infiltration': 'Rare Attack',
    'Heartbleed': 'Rare Attack'
}

# Creating a new column 'Attack Type' in the DataFrame based on the attack_map dictionary
data['Attack Type'] = data['Label'].map(attack_map)

In [ ]:
data['Attack Type'].value_counts()

,count
Attack Type,
BENIGN,2096484
DoS,193748
DDoS,128016
Port Scan,90819
Brute Force,9152
Web Attack,2143
Bot,1953
Rare Attack,47


In [ ]:
data = data[data['Attack Type'] != 'Rare Attack']

In [ ]:
data.drop('Label', axis = 1, inplace = True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Attack Number'] = le.fit_transform(data['Attack Type'])

print(data['Attack Number'].unique())

[0 4 2 6 1 5 3]


In [ ]:
# Printing corresponding attack type for each encoded value
encoded_values = data['Attack Number'].unique()
for val in sorted(encoded_values):
    print(f"{val}: {le.inverse_transform([val])[0]}")


0: BENIGN
1: Bot
2: Brute Force
3: DDoS
4: DoS
5: Port Scan
6: Web Attack


In [ ]:
import joblib

joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [ ]:

# Printing the unique value count
indent = '{:<3} {:<30}: {}'
print('Unique value count for: ')
for i, feature in enumerate(list(data.columns)[:-1], start = 1):
    print(indent.format(f'{i}.', feature, data[feature].nunique()))

Unique value count for: 
1.  Destination Port              : 53805
2.  Flow Duration                 : 1050855
3.  Total Fwd Packets             : 1413
4.  Total Backward Packets        : 1731
5.  Total Length of Fwd Packets   : 17913
6.  Total Length of Bwd Packets   : 64690
7.  Fwd Packet Length Max         : 5279
8.  Fwd Packet Length Min         : 384
9.  Fwd Packet Length Mean        : 99681
10. Fwd Packet Length Std         : 253866
11. Bwd Packet Length Max         : 4836
12. Bwd Packet Length Min         : 583
13. Bwd Packet Length Mean        : 147599
14. Bwd Packet Length Std         : 248853
15. Flow Bytes/s                  : 1593863
16. Flow Packets/s                : 1240119
17. Flow IAT Mean                 : 1166267
18. Flow IAT Std                  : 1056605
19. Flow IAT Max                  : 580273
20. Flow IAT Min                  : 136315
21. Fwd IAT Total                 : 493094
22. Fwd IAT Mean                  : 737693
23. Fwd IAT Std                   : 700282

In [ ]:
cols_to_drop = [
    'Fwd Avg Bytes/Bulk',
    'Fwd Avg Packets/Bulk',
    'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk',
    'Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate'
]

data = data.drop(columns=cols_to_drop)

In [ ]:
data.shape

(2522315, 74)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2522315 entries, 0 to 2830742
Data columns (total 74 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow 

In [ ]:
data['Fwd Header Length'].head(), data['Fwd Header Length.1'].head(),

(0     20
 1    368
 2    336
 3    560
 4    304
 Name: Fwd Header Length, dtype: int64,
 0     20
 1    368
 2    336
 3    560
 4    304
 Name: Fwd Header Length.1, dtype: int64)

In [ ]:
data = data.drop(columns=['Fwd Header Length.1'])

In [ ]:
data.to_parquet('CICIDS_PIN_Cleaned.parquet')

In [ ]:
import pandas as pd
path = "/content/drive/MyDrive/CICIDS_PIN_Cleaned.parquet"
df = pd.read_parquet(path)



In [ ]:
data = pd.read_parquet('CICIDS_PIN_Cleaned.parquet')

FileNotFoundError: [Errno 2] No such file or directory: 'CICIDS_PINDATA.parquet'

In [ ]:
df['Attack Type'].value_counts()

,count
Attack Type,
BENIGN,2096484
DoS,193748
DDoS,128016
Port Scan,90819
Brute Force,9152
Web Attack,2143
Bot,1953


# Correlation

In [ ]:
X = df.drop(columns=['Attack Type', 'Attack Number'])
y = df['Attack Number']

In [ ]:
corr_matrix = X.corr().abs()

In [ ]:
import numpy as np
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

In [ ]:
len(to_drop)

30

In [ ]:
to_drop

['Total Backward Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Std',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow IAT Max',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Bwd IAT Min',
 'Fwd Packets/s',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'SYN Flag Count',
 'CWE Flag Count',
 'ECE Flag Count',
 'Average Packet Size',
 'Avg Fwd Segment Size',
 'Avg Bwd Segment Size',
 'Subflow Fwd Packets',
 'Subflow Fwd Bytes',
 'Subflow Bwd Packets',
 'Subflow Bwd Bytes',
 'Active Min',
 'Idle Mean',
 'Idle Max',
 'Idle Min']

In [ ]:
X = X.drop(columns=to_drop)

In [ ]:
X.shape, y.shape

((2522315, 41), (2522315,))

In [ ]:
df['Attack Type'].value_counts()

,count
Attack Type,
BENIGN,2096484
DoS,193748
DDoS,128016
Port Scan,90819
Brute Force,9152
Web Attack,2143
Bot,1953


In [ ]:
# Combine X and y back temporarily
df_filtered = X.copy()
df_filtered['Attack Number'] = y

# Separate BENIGN and ATTACKS
benign = df_filtered[df_filtered['Attack Number'] == 0].sample(
    n=200000,
    random_state=42
)

attack = df_filtered[df_filtered['Attack Number'] != 0]

# Combine back
df_balanced = pd.concat([benign, attack])

# Shuffle (VERY IMPORTANT)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Split again
X_final = df_balanced.drop('Attack Number', axis=1)
y_final = df_balanced['Attack Number']

In [ ]:
X_final.shape

(625831, 41)

In [ ]:
X_no_port = X_final.drop(columns=['Destination Port'])

# retrain model

In [ ]:
X_no_port

,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,...,URG Flag Count,Down/Up Ratio,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Idle Std
0,223,2,64,32,32,32.000000,48,48,7.174888e+05,17937.219730,...,0,1,-1,-1,1,40,0.000000,0.000000,0,0.000000
1,76,1,0,0,0,0.000000,6,6,7.894737e+04,26315.789470,...,0,1,29200,0,0,40,0.000000,0.000000,0,0.000000
2,13,1,0,0,0,0.000000,0,0,0.000000e+00,153846.153800,...,0,1,5692,63190,0,20,0.000000,0.000000,0,0.000000
3,2,1,0,0,0,0.000000,6,6,3.000000e+06,1000000.000000,...,0,1,29200,0,0,40,0.000000,0.000000,0,0.000000
4,30707,1,60,60,60,60.000000,88,88,4.819748e+03,65.131729,...,0,1,-1,-1,0,32,0.000000,0.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625826,5548,2,0,0,0,0.000000,0,0,0.000000e+00,360.490267,...,0,0,252,-1,0,32,0.000000,0.000000,0,0.000000
625827,99491848,7,415,409,0,59.285714,2896,0,1.207134e+02,0.140715,...,0,1,274,235,2,20,1001.000000,0.000000,1001,0.000000
625828,85930046,4,377,371,0,94.250000,5792,0,1.393226e+02,0.116374,...,0,1,0,235,1,20,0.000000,0.000000,0,0.000000
625829,50,2,4,2,2,2.000000,6,6,3.200000e+05,80000.000000,...,0,1,1024,0,1,24,0.000000,0.000000,0,0.000000


In [ ]:
y_final.value_counts()

,count
Attack Number,
0,200000
4,193748
3,128016
5,90819
2,9152
6,2143
1,1953


In [ ]:
y_final.shape

(625831,)

In [ ]:
from sklearn.model_selection import train_test_split

X_trainF, X_testF, y_trainF, y_testF = train_test_split(
    X_no_port,
    y_final,
    test_size=0.2,
    stratify=y_final,   # 🔥 VERY IMPORTANT
    random_state=42
)

In [ ]:
X_trainF.to_parquet("X_train.parquet")
X_testF.to_parquet("X_test.parquet")

y_trainF.to_frame(name="Label").to_parquet("y_train.parquet")
y_testF.to_frame(name="Label").to_parquet("y_test.parquet")

In [ ]:
# Load data
import pandas as pd

X_train = pd.read_parquet("X_train.parquet")
X_test = pd.read_parquet("X_test.parquet")

y_train = pd.read_parquet("y_train.parquet")["Label"]
y_test = pd.read_parquet("y_test.parquet")["Label"]



In [ ]:
print("Train distribution:\n", y_train.value_counts(normalize=True))
print("\nTest distribution:\n", y_test.value_counts(normalize=True))

Train distribution:
 Attack Number
0    0.319576
4    0.309585
3    0.204554
5    0.145117
2    0.014625
6    0.003423
1    0.003120
Name: proportion, dtype: float64

Test distribution:
 Attack Number
0    0.319573
4    0.309586
3    0.204551
5    0.145118
2    0.014620
6    0.003427
1    0.003124
Name: proportion, dtype: float64


In [ ]:
import json

features = list(X_train.columns)

with open("features.json", "w") as f:
    json.dump(features, f)

In [ ]:
features

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,          # 🔥 limit tree depth
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

In [ ]:
import joblib

joblib.dump(rf, "rf_model.pkl")

In [ ]:
from sklearn.metrics import classification_report

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
from sklearn.metrics import f1_score

f1_macro = f1_score(y_test, y_pred, average='macro')
print(f1_macro)

In [ ]:
print(classification_report(y_train, rf.predict(X_train)))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

print(classification_report(y_test, lr.predict(X_test)))

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=1,
    random_state=42
)

xgb.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print("XGBoost Results:\n")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_xgb)
print(cm)

In [ ]:
import joblib

joblib.dump(xgb, "xgb_model.pkl")

In [ ]:
import joblib

model = joblib.load("xgb_model.pkl")   # or rf_model.pkl

In [ ]:
sample = X_test.sample(3)

pred = model.predict(sample)
proba = model.predict_proba(sample)

for i in range(len(sample)):
    idx = sample.index[i]

    print("Input:\n", sample.iloc[i])

    print("Actual:", label_map[y_test.loc[idx]])
    print("Predicted:", label_map[pred[i]])
    print("Confidence:", max(proba[i]))

    print("-"*50)

In [ ]:
errors = 0

for i in range(len(sample)):
    idx = sample.index[i]

    actual = y_test.loc[idx]
    predicted = pred[i]

    if actual != predicted:
        errors += 1
        print("❌ WRONG")
        print("Actual:", label_map[actual])
        print("Predicted:", label_map[predicted])
        print("-"*30)

print("Total errors:", errors)

In [ ]:
sample = X_test.sample(5).copy()

# Original predictions
pred_original = model.predict(sample)

# Add noise
import numpy as np
noise = np.random.normal(0, 0.05, sample.shape)
sample_noisy = sample + noise

# Noisy predictions
pred_noisy = model.predict(sample_noisy)

# Compare
for i in range(len(sample)):
    print("Original Prediction:", pred_original[i])
    print("Noisy Prediction   :", pred_noisy[i])
    print("-"*40)